### Basic HuggingFace API

1. Load Model / Tokenizer.
2. Run Simple Inference
3. Run Simple Training

In [2]:
# Load Model and Tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_id = "Qwen/Qwen2-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
# Run Simple Inference

prompt = "Give me a step-by-step preparation plan for Research Engineer interviews at frontier AI labs."
messages = [
    {"role": "system", "content": "You are a helpful interview preparation coach."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(text)

model_inputs = tokenizer([text], return_tensors="pt").to(device)
print(model_inputs)
print(model_inputs["input_ids"].shape) # B x L

generated_ids = model.generate(**model_inputs, max_new_tokens=512) # B x Genaration
generated_ids = [
    output_ids[len(input_ids):] for output_ids, input_ids in zip(generated_ids, model_inputs.input_ids)
    ]
print(generated_ids)

response_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print(response_text)


<|im_start|>system
You are a helpful interview preparation coach.<|im_end|>
<|im_start|>user
Give me a step-by-step preparation plan for Research Engineer interviews at frontier AI labs.<|im_end|>
<|im_start|>assistant

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,   7128,  17975,
           7247,     13, 151645,    198, 151644,    872,    198,  35127,    752,
            264,   3019,  14319,  29208,  17975,   3119,    369,   8319,  28383,
          19344,    518,  48000,  15235,  49948,     13, 151645,    198, 151644,
          77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
torch.Size([1, 38])
[tensor([97191,   369,  8319, 28383, 19344,   518, 57318, 15235, 40640, 17601,
         3807,  1376,  7354,   311,  5978,   498,  2299,  1632, 21334,  7212,
          323,  5527,   311, 34783,   697,  7361,   323, 11449,    13,  5692,
      

In [10]:
output = model(**model_inputs)

In [21]:
print(output.past_key_values.layers[0].__dict__)

{'keys': tensor([[[[-7.9062e+00, -2.1094e+00, -5.8750e+00,  ...,  3.2250e+01,
           -1.2900e+02,  7.0500e+01],
          [-1.0188e+01, -7.7188e+00, -7.3750e+00,  ...,  3.4500e+01,
           -1.3000e+02,  6.9500e+01],
          [-1.6094e+00, -9.1875e+00, -6.6875e+00,  ...,  3.6500e+01,
           -1.3000e+02,  7.0000e+01],
          ...,
          [ 1.0250e+01,  7.6875e+00,  1.5625e-02,  ...,  3.2250e+01,
           -1.2900e+02,  7.0500e+01],
          [ 7.7500e+00,  9.3125e+00,  3.4688e+00,  ...,  3.4250e+01,
           -1.3000e+02,  7.0000e+01],
          [-3.1875e+00,  5.4375e+00,  5.9375e+00,  ...,  3.6500e+01,
           -1.3000e+02,  7.0000e+01]],

         [[-8.6250e+00, -1.0703e+00,  5.7500e+00,  ...,  5.8500e+01,
            1.1100e+02, -1.2550e+02],
          [ 3.9000e+01, -1.0438e+01,  5.8750e+00,  ...,  6.1250e+01,
            1.1250e+02, -1.2400e+02],
          [ 4.8250e+01, -1.5938e+01,  6.5000e+00,  ...,  6.6500e+01,
            1.1600e+02, -1.2100e+02],
          .

In [52]:
# Run Batched Inference

batch_messages = [
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is backpropagation?"}
    ],
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is gradient clipping?"}
    ]
]

texts = [
    tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=True
    )
    for chat in batch_messages
]

model_inputs = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,
    truncation=True
).to(device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=128
)

responses = []
for i in range(len(texts)):
    prompt_len = model_inputs["attention_mask"][i].sum().item()
    new_tokens = generated_ids[i, prompt_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    responses.append(text)

for i, r in enumerate(responses):
    print(f"Response {i}: {r}")

Response 0: Backpropagation is an algorithm used in artificial neural networks to train the network by adjusting the weights of the connections between the neurons. It is a type of gradient descent optimization method that allows the network to learn from its mistakes and improve its performance over time.

The basic idea behind backpropagation is to calculate the error made by the network for each input-output pair, and then propagate this error backward through the network using a chain rule to find the change in weight for each neuron in the network. This process is repeated until the network has learned to make accurate predictions on new data inputs.

There are two main types of backpropagation: vanilla
Response 1: 
Gradient clipping is a technique used in machine learning and deep learning to prevent gradients from exploding or vanishing during training. It involves limiting the magnitude of the gradients at each step, which helps to prevent the model from getting stuck in local 

In [2]:
# Build Simple Chat Bot

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_id = "Qwen/Qwen2-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id)


messages = [
    {"role": "system", "content": "You are a helpful assintant."},
]

while True:
    user_prompt = input("User: ")
    messages.append({"role": "user", "content": user_prompt})
    full_chat = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([full_chat], return_tensors="pt").to(device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=128
    )
    new_tokens = generated_ids[:, model_inputs.input_ids.shape[1]:][0]
    assistant_reply = tokenizer.decode(new_tokens, skip_special_tokens=True)
    print("Assistant reply: \n", assistant_reply)
    messages.append({"role": "assistant", "content": assistant_reply})


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Assistant reply: 
 Hello! How can I assist you today?
Assistant reply: 
 Backpropagation is an algorithm used in machine learning to train neural networks. It is named after its creator, Jacek Bogoński, who proposed it in 1986.
The basic idea behind backpropagation is that the error between the network's predictions and the actual outputs should be propagated backwards through the network to adjust the weights of each neuron. This process continues until the error is minimized.
There are several variations of backpropagation, including stochastic gradient descent (SGD) and batch gradient descent. SGD involves randomly selecting a subset of data points for each update, while batch gradient descent updates all neurons at once.

Assistant reply: 
 Tajikistan is a landlocked country located in Central Asia, bordered by Afghanistan to the west, China to the north, and Pakistan and Uzbekistan to the east. The capital and largest city is Dushanbe.
Tajikistan has a diverse landscape with high 

KeyboardInterrupt: 

In [6]:
# Simple SFT training


from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig

model_name = "Qwen/Qwen2.5-0.5B-Instruct"   # example small chat model

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

train_data = Dataset.from_list([
    {
        "messages": 
        [
            {"role": "system", "content": "You are a helpful interview coach."},
            {"role": "user", "content": "How do I prepare for RE interviews?"},
            {"role": "assistant", "content": "Start with coding, ML systems, and research taste."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a helpful interview coach."},
            {"role": "user", "content": "What should I study for transformers?"},
            {"role": "assistant", "content": "Study attention, tokenization, scaling, and inference."}
        ]
    }
])

training_args = SFTConfig(
    output_dir="./sft-model",
    per_device_train_batch_size=2,
    learning_rate=2e-5,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=100,
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
)

trainer.train()
trainer.save_model("./sft-model-final")
tokenizer.save_pretrained("./sft-model-final")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

KeyError: 'text'

### Implement Different Inference strategies from Scratch
1. Greedy Decoding: Vanilla + KV cache
2. Top-k sampling
3. Top-p sampling
4. Beam-Search
5. Speculative decoding

In [170]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch 
from functools import partial 

class CustomWrapper:
    def __init__(self, model_id: str, max_completions_len: int = 512, temperature: float=0.8, top_p: float = 0.95, top_k: int = 20):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = AutoModelForCausalLM.from_pretrained(model_id).to(self.device)
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.generation_kwargs = {"max_completions_len": max_completions_len, "temperature": temperature, "top_p": top_p, "top_k": top_k}
        self.eos_token_id = self.model.generation_config.eos_token_id
        if not isinstance(self.eos_token_id, list):
            self.eos_token_id = [self.eos_token_id]
        
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.generation_algorithms = {
            "hf_greedy": self.hf_greedy,
            "greedy_vanilla": self.greedy_vanilla,
            "greedy_sampling": partial(self.custom_generation, sampling_method=self.greedy_sampling),
            "topp_sampling": partial(self.custom_generation, sampling_method=self.topp_sampling),
            "topk_sampling": partial(self.custom_generation, sampling_method=self.topk_sampling),
        }

    @torch.no_grad()
    def generate(self, prompt, algorithm="greedy_sampling"):
        self.model.eval()
        tokenized_prompt = self.tokenizer(prompt, return_tensors="pt")
        model_inputs = {"input_ids": tokenized_prompt.input_ids.to(self.device),
                        "attention_mask": tokenized_prompt.attention_mask.to(self.device)
                    }
        generated_ids = self.generation_algorithms[algorithm](model_inputs)
        return self.tokenizer.decode(generated_ids, skip_special_tokens=True)

    @torch.no_grad() 
    def hf_greedy(self, model_inputs):
        output_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=self.generation_kwargs["max_completions_len"],
            do_sample=False,
            use_cache=True,
            pad_token_id=self.tokenizer.pad_token_id,
        )
        generated_ids = output_ids[0, model_inputs["input_ids"].shape[1]:]
        return generated_ids
        
    @torch.no_grad()
    def greedy_vanilla(self, model_inputs):
        generated_ids = []
        for _ in range(self.generation_kwargs["max_completions_len"]):
            next_token_id = self.model(**model_inputs).logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated_ids.append(next_token_id.item())
            if generated_ids[-1] in self.eos_token_id:
                break
            model_inputs["input_ids"] = torch.cat((model_inputs["input_ids"], next_token_id), dim=-1)
            model_inputs["attention_mask"] = torch.cat(
                (
                    model_inputs["attention_mask"], 
                    torch.ones(
                        (model_inputs["attention_mask"].shape[0], 1),
                        dtype=model_inputs["attention_mask"].dtype,
                        device=model_inputs["attention_mask"].device,
                        )
                ), 
                dim=-1
            )
        return generated_ids

    @torch.no_grad()
    def custom_generation(self, model_inputs, sampling_method):
        # model_inputs should contain input_ids and attention_masks: size 1 x L 
        generated_ids = []
        output = self.model(**model_inputs, use_cache=True)
        attention_mask = model_inputs["attention_mask"]
        past_key_values = output.past_key_values 
        next_token_logits = output.logits[:, -1, :] # 1 x L x V -> 1 x V
        for _ in range(self.generation_kwargs["max_completions_len"]):
            next_token_id = sampling_method(next_token_logits / self.generation_kwargs["temperature"])
            generated_ids.append(next_token_id.detach().item())
            if generated_ids[-1] in self.eos_token_id:
                break
            attention_mask = torch.cat(
                (
                    attention_mask, 
                    torch.ones(
                        attention_mask.shape[0], 
                        1, 
                        dtype=attention_mask.dtype, 
                        device=attention_mask.device
                        )
                 ),
                    dim=-1
            )
            position_ids = attention_mask.sum(dim=-1, keepdim=True)-1
            output = self.model(
                input_ids=next_token_id, 
                attention_mask=attention_mask, 
                use_cache=True, 
                past_key_values=past_key_values, 
                position_ids=position_ids)
            next_token_logits = output.logits[:, -1, :]
            past_key_values = output.past_key_values
        return generated_ids
    
    @torch.no_grad()
    def greedy_sampling(self, logits):
        return logits.argmax(dim=-1, keepdim=True)
    
    @torch.no_grad()
    def topk_sampling(self, logits):
        topk_logits, topk_token_ids = logits.topk(k=self.generation_kwargs["top_k"], dim=-1)
        topk_probs = topk_logits.softmax(dim=-1)
        sampled_idx = torch.multinomial(topk_probs, num_samples=1)
        next_token_ids = topk_token_ids.gather(index=sampled_idx, dim=-1)
        return next_token_ids    
    
    @torch.no_grad()
    def topp_sampling(self, logits):
        probs = logits.softmax(dim=-1)
        sorted_probs, sorted_indices = probs.sort(dim=-1, descending=True)
        cum_probs = sorted_probs.cumsum(dim=-1)
        topp_mask = cum_probs > self.generation_kwargs["top_p"]
        # include one extra token to make cum sum > topp
        topp_mask[:, 1:] = topp_mask[:, :-1].clone()
        topp_mask[:, 0] = False
        sorted_probs = sorted_probs.masked_fill(topp_mask, 0.0)
        sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)
        sampled_sorted_idx = torch.multinomial(sorted_probs, num_samples=1)
        next_token_ids = sorted_indices.gather(index=sampled_sorted_idx, dim=-1)
        return next_token_ids
    
    @torch.no_grad()
    def beam_search(self, model_inputs):
        generated_ids = []
        output = self.model(**model_inputs, use_cache=True)

In [171]:
import time

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "How to create a personal mission statement?"}
]

custom_api = CustomWrapper(model_id = "Qwen/Qwen2.5-0.5B-Instruct", max_completions_len=20)

text = custom_api.tokenizer.apply_chat_template(messages, tokenize=False,
        add_generation_prompt=True)
print(f"Input Prompt Text: \n{text}")

for algo in ["hf_greedy", "greedy_vanilla", "greedy_sampling", "topk_sampling", "topp_sampling"]:
    start_time = time.time()
    generated_text = custom_api.generate(prompt=text, algorithm=algo)
    print(f"=============Algorithm  {algo}  =========\n")
    print(f"===Time {time.time()-start_time}=======\n")
    print(generated_text, '\n')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Input Prompt Text: 
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
How to create a personal mission statement?<|im_end|>
<|im_start|>assistant

=============Algorithm  hf_greedy  =========

===Time 2.3469717502593994=======

Creating a personal mission statement is an important step in setting your career goals and defining what you want to 

=============Algorithm  greedy_vanilla  =========

===Time 5.325206995010376=======

Creating a personal mission statement is a great way to clarify your goals and values, and to help you 

=============Algorithm  greedy_sampling  =========

===Time 0.6143627166748047=======

Creating a personal mission statement is a great way to clarify your goals, values, and aspirations. Here 

=============Algorithm  topk_sampling  =========

===Time 0.6217780113220215=======

Creating a personal mission statement is a great way to define what you're passionate about and what kind of 

=============Algorithm  topp_sampling  =======

In [167]:
x = torch.randn(2, 4)
mask = torch.tensor([True, False, True, False])
x = x.masked_fill(mask, -1)
x


tensor([[-1.0000, -0.9927, -1.0000,  1.2124],
        [-1.0000,  0.6468, -1.0000, -1.6282]])